In [55]:
import numpy as np
import pickle
from glob import glob
import pandas as pd
import matplotlib.pyplot as plt
import os.path as op
import os
from xcp_d.interfaces.workbench import CiftiMath, CiftiParcellateWorkbench, CiftiConvert

import nilearn
from nipype.interfaces.workbench.base import WBCommand
from nipype.interfaces.base import TraitedSpec, File, CommandLineInputSpec, traits

import nibabel as nb
os.environ["PATH"] = "/global/homes/m/mphagen/software/workbench/bin_linux64:" + os.environ["PATH"]

In [2]:
path_prefix = '/pscratch/sd/m/mphagen'

In [3]:
dataset_path = op.join(path_prefix, 'hcp-functional-connectivity') 
sub_path = '107725/MNINonLinear/Results/rfMRI_REST1_LR/rfMRI_REST1_LR_Atlas_MSMAll_hp2000_clean.dtseries.nii' 

In [4]:
class _CiftiStatOutputSpec(TraitedSpec):
    """Output specification for the CiftiParcellateWorkbench command."""

    out_file = File(exists=True, desc="output file")


In [5]:
class _CiftiStatInputSpec(CommandLineInputSpec):
    """Output specification for ConvertAffine."""

    in_file = File(
        position=1,
        argstr='%s',
        exists=True,
        mandatory=True,
        desc='input file',
    )
    
    reduce = traits.Enum(
        'TSNR', 
        mandatory=False,
        argstr=' %s',
        position=2,
        desc='Which mapping to parcellate (integer, ROW, or COLUMN)',
    )
    
    # reduce = traits.Enum(
    #     'TSNR', 
    #     mandatory=False,
    #     argstr='-reduce %s',
    #     position=2,
    #     desc='Which mapping to parcellate (integer, ROW, or COLUMN)',
    # )
    column = traits.Enum(
        '1', 
        mandatory=False,
        argstr='-column %s',
        position=3,
        desc='Which mapping to parcellate (integer, ROW, or COLUMN)',
    )

    roi = traits.Enum(
        'show-map-name', 
        mandatory=False,
        argstr='-%s',
        position=4,
        desc='Which mapping to parcellate (integer, ROW, or COLUMN)',
    )
    out_file = File(
        name_source='in_file',
        name_template='%s_stat.nii',
        keep_extension=False,
        argstr='%s',
        desc='Output cifti file',
    )

In [6]:
class CiftiStat(WBCommand):
    """Interface for wb_command's -convert-affine command."""

    input_spec = _CiftiStatInputSpec
    output_spec = _CiftiStatOutputSpec

    _cmd = "wb_command -cifti-reduce"


wb_command -cifti-stats rfMRI_REST1_LR_Atlas_MSMAll_hp2000_clean.dtseries.nii -reduce TSNR

In [7]:
voxel_cifti = op.join('/pscratch/sd/m/mphagen/hcp-functional-connectivity/107725/MNINonLinear/Results/rfMRI_REST1_LR', 
    'rfMRI_REST1_LR_Atlas_MSMAll_hp2000_clean.dtseries.nii') 

In [30]:
voxel_nifti = op.join('/pscratch/sd/m/mphagen/hcp-functional-connectivity/107725/MNINonLinear/Results/rfMRI_REST1_LR', 
                      'rfMRI_REST1_LR_hp2000_clean.nii.gz')

In [8]:
ciftistat = CiftiStat()


In [9]:
# (np.mean(cifti, axis=0) / np.std(cifti, axis=0))[75:80]

In [10]:
# np.mean(cifti[:,1]) / np.std(cifti[:,1]) 

In [11]:
# ts_file  = op.join('/global/u1/m/mphagen/functional-connectivity/connectome-comparison/scripts/processing', 
# 'sub-202113_task-rest_dir-RL_run-1_space-fsLR_seg-4S156Parcels_den-91k_stat-mean_timeseries.ptseries.nii') 

In [12]:
ciftistat.inputs.in_file = voxel_cifti
ciftistat.inputs.reduce = 'TSNR'
ciftistat.set_default_terminal_output = 'file_stdout'
ciftistat.inputs.out_file = 'out_file.dscalar.nii'

In [13]:
ciftistat.cmdline

'wb_command -cifti-reduce /pscratch/sd/m/mphagen/hcp-functional-connectivity/107725/MNINonLinear/Results/rfMRI_REST1_LR/rfMRI_REST1_LR_Atlas_MSMAll_hp2000_clean.dtseries.nii  TSNR out_file.dscalar.nii'

In [14]:
import nipype
import logging
logging.getLogger('nipype.workflow').setLevel(0)

results = ciftistat.run() 

In [46]:
cifticonvert = CiftiConvert()
cifticonvert.inputs.in_file = 'out_file.dscalar.nii'
cifticonvert.inputs.target = "to"
cifticonvert.cmdline

'wb_command -cifti-convert -to-nifti out_file.dscalar.nii out_file_converted.nii.gz'

In [15]:
vox = nb.load(voxel_cifti) 
vox_data = vox.get_fdata() 
vox_data.shape

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


(1200, 91282)

In [31]:
vox_nifti = nb.load(voxel_nifti)

In [33]:
nifti_data = vox_nifti.get_fdata() 

In [34]:
nifti_data.shape

(91, 109, 91, 1200)

In [16]:
tsnr_array = np.mean(vox_data, axis=0) / np.std(vox_data, axis=0)

In [59]:
import matplotlib.pyplot as plt

In [66]:
tsnr_cifti = nb.load('out_file.dscalar.nii')

In [51]:
tsnr_nifti = nb.load('out_file_converted.nii.gz')

In [67]:
tsnr_nifti.get_fdata().shape

(32767, 3, 1, 1)

In [68]:
tsnr_cifti.get_fdata().shape

(1, 91282)

In [69]:
tsnr_cifti.header.get_axis(0).get_element(0) 

(np.str_('TSNR'), {})

In [70]:
tsnr_cifti.files_types

(('image', '.nii'),)

In [71]:
tsnr_cifti.header.get_axis(1).affine

array([[  -2.,    0.,    0.,   90.],
       [   0.,    2.,    0., -126.],
       [   0.,    0.,    2.,  -72.],
       [   0.,    0.,    0.,    1.]])

In [74]:
tsnr_nifti = nb.Nifti2Image(tsnr_cifti.get_fdata() , tsnr_cifti.header.get_axis(1).affine)


In [75]:
tsnr_nifti.shape

(1, 91282)

In [27]:
tsnr_cifti.nifti_header.get_best_affine()

array([[-1.,  0.,  0.,  0.],
       [ 0.,  1.,  0., -0.],
       [ 0.,  0.,  1., -0.],
       [ 0.,  0.,  0.,  1.]])

In [61]:
tsnr_nifti.shape

(32767, 3, 1, 1)

In [226]:
tsnr.header.get_data_shape()

AttributeError: 'Cifti2Header' object has no attribute 'get_data_shape'

In [219]:
dir(tsnr.nifti_header) 

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__non_callable_proto_members__',
 '__parameters__',
 '__protocol_attrs__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_chk_bitpix',
 '_chk_datatype',
 '_chk_eol_check',
 '_chk_magic',
 '_chk_offset',
 '_chk_pixdims',
 '_chk_qfac',
 '_chk_qform_code',
 '_chk_sform_code',
 '_chk_sizeof_hdr',
 '_chk_xform_code',
 '_clean_after_mapping',
 '_data_type_codes',
 '_field_recoders',
 '_get_checks',
 '_is_protocol',
 '_is_runtime_protocol',
 '_slice_time_order',
 '_structarr',
 'as_analyze_map',
 'as_byteswapped',
 'b

In [216]:
plotting.plot_epi(tsnr_nifti, cbar_tick_format="%i")


DimensionError: Input data has incompatible dimensionality: Expected dimension is 3D and you provided a 2D image. See https://nilearn.github.io/stable/manipulating_images/input_output.html.

In [213]:
#to make new nifti from array, doesn't currently plot properly - might be matplotlib issue
#tsnr_nifti = nb.Nifti2Image(tsnr.get_fdata() , np.eye(4,4), header=tsnr.nifti_header)
plotting.plot_roi(tsnr_nifti) 

DimensionError: Input data has incompatible dimensionality: Expected dimension is 3D and you provided a 2D image. See https://nilearn.github.io/stable/manipulating_images/input_output.html.

In [177]:
# tsnr_nifti = nb.load('outfile.dscalar.nii.gz')

In [183]:
print(tsnr_nifti.header ) 

<class 'nibabel.nifti1.Nifti1Header'> object, endian='<'
sizeof_hdr      : 348
data_type       : np.bytes_(b'')
db_name         : np.bytes_(b'')
extents         : 0
session_error   : 0
regular         : np.bytes_(b'')
dim_info        : 0
dim             : [    4 32767     3     1     1     1     1     1]
intent_p1       : 0.0
intent_p2       : 0.0
intent_p3       : 0.0
intent_code     : none
datatype        : float32
bitpix          : 32
slice_start     : 0
pixdim          : [1. 1. 1. 1. 1. 1. 1. 1.]
vox_offset      : 0.0
scl_slope       : nan
scl_inter       : nan
slice_end       : 0
slice_code      : unknown
xyzt_units      : 10
cal_max         : 0.0
cal_min         : 0.0
slice_duration  : 0.0
toffset         : 0.0
glmax           : 0
glmin           : 0
descrip         : np.bytes_(b'Connectome Workbench, version 2.1.0')
aux_file        : np.bytes_(b'')
qform_code      : mni
sform_code      : mni
quatern_b       : 0.0
quatern_c       : 0.0
quatern_d       : 0.0
qoffset_x       : 0.0


In [149]:
plotting.plot_stat_map(tsnr_nifti ) 

TypeError: _warn() got an unexpected keyword argument 'skip_file_prefixes'

In [109]:
tsnr.get_fdata()

memmap([[ 58.55736542,  96.15618896, 115.27719879, ...,  13.24809456,
          15.73408699,  13.9086628 ]], shape=(1, 91282))

In [91]:
dir(tsnr) 

['__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_compressed_suffixes',
 '_data_cache',
 '_dataobj',
 '_fdata_cache',
 '_filemap_from_iobase',
 '_header',
 '_meta_sniff_len',
 '_nifti_header',
 '_sniff_meta_for',
 'dataobj',
 'extra',
 'file_map',
 'files_types',
 'filespec_to_file_map',
 'from_bytes',
 'from_file_map',
 'from_filename',
 'from_image',
 'from_stream',
 'from_url',
 'get_data',
 'get_data_dtype',
 'get_fdata',
 'get_filename',
 'header',
 'header_class',
 'in_memory',
 'instance_to_filename',
 'load',
 'make_file_map',
 'makeable',
 'ndim',
 'nifti_header',
 'path_maybe_image',
 'rw',
 'set_data_dtype

In [71]:
dir(tsnr.nifti_header) 

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__non_callable_proto_members__',
 '__parameters__',
 '__protocol_attrs__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_chk_bitpix',
 '_chk_datatype',
 '_chk_eol_check',
 '_chk_magic',
 '_chk_offset',
 '_chk_pixdims',
 '_chk_qfac',
 '_chk_qform_code',
 '_chk_sform_code',
 '_chk_sizeof_hdr',
 '_chk_xform_code',
 '_clean_after_mapping',
 '_data_type_codes',
 '_field_recoders',
 '_get_checks',
 '_is_protocol',
 '_is_runtime_protocol',
 '_slice_time_order',
 '_structarr',
 'as_analyze_map',
 'as_byteswapped',
 'b

In [67]:
dir(tsnr)

['__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_compressed_suffixes',
 '_data_cache',
 '_dataobj',
 '_fdata_cache',
 '_filemap_from_iobase',
 '_header',
 '_meta_sniff_len',
 '_nifti_header',
 '_sniff_meta_for',
 'dataobj',
 'extra',
 'file_map',
 'files_types',
 'filespec_to_file_map',
 'from_bytes',
 'from_file_map',
 'from_filename',
 'from_image',
 'from_stream',
 'from_url',
 'get_data',
 'get_data_dtype',
 'get_fdata',
 'get_filename',
 'header',
 'header_class',
 'in_memory',
 'instance_to_filename',
 'load',
 'make_file_map',
 'makeable',
 'ndim',
 'nifti_header',
 'path_maybe_image',
 'rw',
 'set_data_dtype

In [22]:
from nilearn import plotting

In [158]:
import matplotlib.pyplot as plt

In [24]:
atlas_path = op.join(path_prefix, 'AtlasPack', 'tpl-fsLR_atlas-4S156Parcels_den-91k_dseg.dlabel.nii') 

In [27]:
atlas_pack = nb.load(atlas_path)

In [ ]:
a

In [51]:
len(atlas_pack.get_fdata()[0]) 

91282

In [54]:
atlas_pack.header.get_axis(1).name[90000]

np.str_('CIFTI_STRUCTURE_THALAMUS_LEFT')

In [32]:
dir(atlas_pack.header) 

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_to_xml_element',
 'copy',
 'from_axes',
 'from_fileobj',
 'from_header',
 'get_axis',
 'get_index_map',
 'mapped_indices',
 'matrix',
 'may_contain_header',
 'number_of_mapped_indices',
 'to_xml',
 'version',
 'write_to']

In [31]:
dir(atlas_pack)

['__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_compressed_suffixes',
 '_data_cache',
 '_dataobj',
 '_fdata_cache',
 '_filemap_from_iobase',
 '_header',
 '_meta_sniff_len',
 '_nifti_header',
 '_sniff_meta_for',
 'dataobj',
 'extra',
 'file_map',
 'files_types',
 'filespec_to_file_map',
 'from_bytes',
 'from_file_map',
 'from_filename',
 'from_image',
 'from_stream',
 'from_url',
 'get_data',
 'get_data_dtype',
 'get_fdata',
 'get_filename',
 'header',
 'header_class',
 'in_memory',
 'instance_to_filename',
 'load',
 'make_file_map',
 'makeable',
 'ndim',
 'nifti_header',
 'path_maybe_image',
 'rw',
 'set_data_dtype

In [30]:
atlas_pack.header

In [136]:

# ciftiparcel = CiftiParcellateWorkbench()
# ciftiparcel.inputs.in_file = '/pscratch/sd/m/mphagen/hcp-functional-connectivity/107725/MNINonLinear/Results/rfMRI_REST1_LR/rfMRI_REST1_LR_Atlas_MSMAll_hp2000_clean.dtseries.nii'
# ciftiparcel.inputs.out_file = 'outfile.ptseries.nii'
# ciftiparcel.inputs.atlas_label =  atlas_path
# ciftiparcel.inputs.direction = 'ROW'
# ciftiparcel.inputs.cor_method = 'TSNR'

# ciftiparcel.cmdline  
# print(ciftiparcel.cmdline)
# ciftiparcel.run()
#     # os.system(cift

wb_command -cifti-parcellate /pscratch/sd/m/mphagen/hcp-functional-connectivity/107725/MNINonLinear/Results/rfMRI_REST1_LR/rfMRI_REST1_LR_Atlas_MSMAll_hp2000_clean.dtseries.nii /pscratch/sd/m/mphagen/AtlasPack/tpl-fsLR_atlas-4S156Parcels_den-91k_dseg.dlabel.nii ROW outfile.ptseries.nii -method TSNR
251105-22:05:26,340 nipype.interface INFO:
	 stderr 2025-11-05T22:05:26.340431:
251105-22:05:26,341 nipype.interface INFO:
	 stderr 2025-11-05T22:05:26.340431:While running:
251105-22:05:26,341 nipype.interface INFO:
	 stderr 2025-11-05T22:05:26.340431:/global/u1/m/mphagen/software/workbench/bin_linux64/../exe_linux64/wb_command -cifti-parcellate /pscratch/sd/m/mphagen/hcp-functional-connectivity/107725/MNINonLinear/Results/rfMRI_REST1_LR/rfMRI_REST1_LR_Atlas_MSMAll_hp2000_clean.dtseries.nii /pscratch/sd/m/mphagen/AtlasPack/tpl-fsLR_atlas-4S156Parcels_den-91k_dseg.dlabel.nii ROW outfile.ptseries.nii -method TSNR
251105-22:05:26,341 nipype.interface INFO:
	 stderr 2025-11-05T22:05:26.340431:


In [137]:
import nibabel as nb

In [138]:
cifti = nb.load('/global/homes/m/mphagen/functional-connectivity/connectome-comparison/scripts/notebooks/exploratory/out_file.pscalar.nii')
cifti_data = cifti.get_fdata(dtype=np.float32)

In [ ]:
results.runtime

In [ ]:
dir(ciftistat)

In [ ]:
ciftistat._list_outputs()

In [ ]:
results.outputs

In [ ]:
dir(results)

In [ ]:
dir(results)

In [ ]:
import nibabel as nb

In [ ]:
ts_files = glob('/pscratch/sd/m/mphagen/hcp-functional-connectivity/derivatives/timeseries/xcpd/sub-*/*ses-1*ptseries.nii')

In [ ]:
sub_list = []
_ = [sub_list.append(ii.split('/')[-2]) for ii in ts_files]

In [ ]:
len(ts_files) 

In [ ]:
ts_files.sort()

In [ ]:
ts_files[10]

In [ ]:
cifti = nb.load(ts_files[10])
cifti_data = cifti.get_fdata(dtype=np.float32)

In [ ]:
#cifti data is zero indexed, so node_7 in 2025-11-04 is actually node 8

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,15) ) 

ax[0].xcorr(cifti_data[:,8], cifti_data[:,8], color='red', maxlags=50)
ax[0].set_ylim([-.4, .7])
ax[0].set_title('Node 7, average acc = ~.9')
# ax[1,1].xcorr(cifti[:,70], cifti[:,70],   color='blue', maxlags=2399) 

# ax[0,1].xcorr(cifti[:,30], cifti[:,70], color='purple', maxlags=2399) 
ax[1].set_title('Node 30, average acc = ~.24')

ax[1].xcorr(cifti_data[:,31], cifti_data[:,31], color='red', maxlags=50) 
# ax[1,0].xcorr(cifti[:,30], cifti[:,30], color='blue', maxlags=2399) 
ax[1].set_ylim([-.4, .7])

plt.savefig('sub-102109_autocorrelation_lag500.png') 


In [ ]:
data = cifti_data

In [ ]:
cifti_data.shape

In [ ]:
meanimg = np.mean(data, axis=3)
stddevimg = np.std(data, axis=3)
tsnr = np.zeros_like(meanimg)
stddevimg_nonzero = stddevimg > 1.0e-3
tsnr[stddevimg_nonzero] = (
    meanimg[stddevimg_nonzero] / stddevimg[stddevimg_nonzero]) 

In [ ]:
for ii in range(0, 99): 
    print(np.mean(cifti_data[:,ii])  / np.std(cifti_data[:,ii]) ) 